# EXP032b — 1D cycle revival scan (state-level + operator-level)

Companion to `EXP033.ipynb`. EXP033 transcribes and verifies the
1D revival catalogue from Dukes (2014), *Quantum state revivals in
quantum walks on cycles*, Results in Physics 4, 189–197; this
notebook **searches** for revivals that the catalogue claims exist,
finds any the catalogue missed, and quantifies the negative cases.

## Notebook process
The search performs a coarse grid search for k=7,9,11,12,13,14,15,16 expecting to find no revivals.
It repeats the search on a 1001 x 360 grid (rho x delta) for the same set in a. fine search looking for revivals - Expect none found
It runs the same fine search for the known revival k values (2,3,4,5,6,8,10) and reports all revivals found.

## Two criteria, two modes

| | Default mode | `--exhaustive` mode |
|---|---|---|
| Criterion | **state-level**, `ψ_N = c·ψ_0` (weaker than projective) | **operator-level**, `U^N = I` (exact) and `U^N = e^{iφ}I` (projective) |
| Grid | `RHO_STEPS × |DELTA_SWEEP|` ≈ 201 × 10 | `--rho-fine × --delta-fine` ≈ 1001 × ~370 |
| Cost | seconds | minutes per k, parallelised across k |
| Output | which steps return the *chosen* initial state | every revival period, plus a worst-residual witness |
| Used for | eyeballing; not citable | the paper's null result and the k=2 correction |

The paper's "172 signatures", "exhaustive grid search" null result,
and "worst residual 2.0" all come from `--exhaustive`. The default
mode is a diagnostic only.

## What `--exhaustive` does, step by step

1. **Skip trivial points.** `ρ ∈ {0,1}` or `δ ∈ {0,π}` ⇒ coin is a
   permutation, walk reduces to a position permutation, classified
   analytically and *never* enters the eigensolve.
2. **Eigensolve once per (ρ, δ).** `U = S·(I ⊗ C)` diagonalised;
   the N-loop is `z *= eigs` (cheap).
3. **Detect.** Exact: `max_j |λ_j^N − 1| < 1e-9`. Projective:
   `max_j |λ_j^N − mean(λ^N)| < 1e-9`. Both confirmed with a fresh
   `eigs**N` at the detection step (v6.1: removes accumulated
   phase drift before `φ` feeds `N_exact = N_proj · ord(φ)`).
4. **Refine.** Snap `(ρ, δ)` to nearest small rational / rational
   multiple of `2π`; re-check at `tol_eig = 1e-12`; confirm with
   `matrix_power` at `tol_mat = 1e-10`. Anything failing refinement
   is reported as `unconfirmed` — **`unconfirmed > 0` is a bug**,
   not a physical result.
5. **Classify.** `trivial` (skipped upstream) / `dukes` (matches
   `DUKES_EXACT`) / `new`. On the known-revival k-set, `new` should
   be empty except for the documented k=2, ρ=1/2, δ∈{π/2,3π/2}
   correction. On the negative k-set, `new` should be empty entirely.

## Scope caveats (unchanged in spirit, updated in numbers)

- **State-level ≠ projective.** The default `full` test is
  `same_up_to_phase(ψ, ψ₀)` for *one* initial state. It does not
  imply `U^N = e^{iφ}I`. Only `--exhaustive` checks the operator.
- **The grid is finite.** Default `--rho-fine 1001` ⇒ spacing
  ≈ 1e-3 in ρ; `--delta-fine 360` plus deduped Dukes values ⇒
  ≈ 370 points in δ, spacing ≈ 1.7e-2. A revival narrower than
  that spacing will be missed. The paper reports the worst
  residual (max `|λ_j^N − 1|` with `|λ_j|=1`, which is 2.0 when
  `λ_j^N = −1`) as the sensitivity witness — not as an error.
- **The δ grid is not the flag value.** `build_delta_grid` appends
  the 10 `DELTA_SWEEP` values to `linspace(0, 2π, delta_steps,
  endpoint=False)` and dedupes. The header prints the *actual*
  grid size (`len(delta_grid)`), which is usually a few larger
  than `--delta-fine`. Quote the printed size in the paper, not
  the flag.

## Running it

Written to disk with `%%writefile EXP032b.py` in the cell above.

### Default mode (state-level, fast — diagnostic only)

```bash
# Single δ, one k, short N, human-readable
python EXP032b.py --k 4 --beta 1.5707963 --N 20 --verbose 0

# Standard δ loop, CSV per δ
python EXP032b.py --delta-sweep --csv sweep.csv

# One δ, full ρ grid, flag anything not in Dukes
python EXP032b.py --rho-steps 201 --N 60 --csv dukes_scan.csv --flag-new

## Run the exhaustive `(ρ, δ)` sweep — optimised (v6)

This cell rewrites `EXP032b.py` with three accuracy-preserving optimisations
over the v5 code that produced `exhaustive_report_full.txt`:

1. **Trivial points are skipped before the eigensolve.** Points with
   `ρ ∈ {0, 1}` or `δ ∈ {0, π, 2π}` (within `1e-6`) are classified
   trivial analytically — they are never sent through
   `np.linalg.eigvals`. This removes the spurious `unconfirmed`
   warnings on k=9 and k=15 (all of which were `ρ = 1.0` endpoint
   roundoff) and makes the negative-case report read "0 confirmed
   revivals", which is the honest statement. The count of skipped
   trivial points is reported per-k as a diagnostic.

2. **Iterative `z *= eigs` instead of `z = eigs ** N`** inside the
   N-loop. For negative-case points all 60 iterations run; this is
   ~60× fewer transcendental operations per point. The difference
   from `eigs ** N` is ~1e-14 at N=60 with |λ|=1, far below
   `TOL_EXACT = 1e-9`, so no accuracy is affected.

3. **BLAS threading is pinned to 1 thread per worker** and
   `--workers` defaults to `os.cpu_count()`. The v5 parallel run
   (`workers: 2`, ~5000s wall, ~1300s per k) was ~25× *slower*
   per k than the serial run, almost certainly because each
   worker's numpy was spawning its own BLAS threads and they were
   fighting for cores. With one BLAS thread per worker and one
   worker per core, the k-loop parallelises cleanly.

### What to run

# Known-revival k-set, full dense grid, 8 workers
# (~minutes; the paper's run is 1001 x ~370)
python EXP032b.py --exhaustive --k 2,3,4,5,6,8,10 \
    --rho-fine 1001 --delta-fine 360 \
    --report exhaustive_known.txt --workers 8

# Negative k-set: expected null, this is the paper's claim
python EXP032b.py --exhaustive --k 7,9,11,13,14,15,16 \
    --rho-fine 1001 --delta-fine 360 \
    --report exhaustive_negative.txt --workers 8

# Full k-set (v6.1 K_SET = 2..16) on 1001 x 360 grid, let worker count auto-size
python EXP032b.py --exhaustive --k-all \
    --report exhaustive_all.txt

# Quick smoke test (~seconds) before committing to a full run
python EXP032b.py --exhaustive --k 4,6 \
    --rho-fine 201 --delta-fine 72 --workers 2

In [1]:

%%writefile EXP032b.py
#!/usr/bin/env python3
# ============================================================
#  EXP032b.py  (v6.1)
#
#  Sweep (rho, delta) for discrete quantum walks on k-cycles,
#  using the coin of Dukes (2014), "Quantum state revivals in
#  quantum walks on cycles".
#
#  v6.1 changes vs v6 (from review):
#    - When a candidate is detected via the iterative
#      z *= eigs form, recompute eigs ** N exactly once to
#      confirm the detection and to pin phi_proj to machine
#      precision.  This removes ~1e-14 accumulated phase drift
#      from phi_proj before it feeds the phase-order
#      derivation N_exact = N_proj * ord(phi).
#    - Report header now shows trivial_skipped from the per-k
#      counter, not from refined_hits (which is always empty
#      for the trivial class by construction).
#    - exact_residual_eig / projective_residual_eig updated
#      to use the iterative form, matching the fast path.
#    - Removed the dead delta == 2*pi branch from
#      is_trivial_point (2*pi is never in the grid; note in
#      comment that delta is periodic with 2*pi in the coin).
#    - Header prints os.cpu_count() and
#      len(os.sched_getaffinity(0)) so cgroup limits are
#      visible on Colab/cloud instances.
#
#  v6 changes vs v5 (all accuracy-preserving):
#    - Skip trivial (rho in {0,1}, delta in {0,pi}) points
#      BEFORE the eigensolve.  They are classified
#      analytically as trivial; running them through eigvals
#      was waste and produced spurious 'unconfirmed' entries
#      at the rho=1.0 endpoint.
#    - Iterative z *= eigs in the N-loop instead of
#      z = eigs ** N.  ~60x fewer transcendental ops per point
#      for negative cases; difference from eigs**N is ~1e-14
#      at N=60 with |lambda|=1, far below TOL_EXACT=1e-9.
#    - BLAS threading pinned to 1 thread per worker and
#      --workers defaults to available cores.
#    - print_exhaustive_report reads refined_hits and
#      unconfirmed as lists of dicts (v5 bug fixed).
#    - K_SET trimmed to [2..16]; --k-all is the canonical
#      way to run the full set.
# ============================================================

import os
# Pin BLAS to a single thread per process BEFORE numpy import.
# Must happen before "import numpy" for OpenBLAS/MKL to honour it.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import csv
import argparse
import sys



import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from math import sqrt, pi, gcd
from fractions import Fraction

# ------------------------------------------------------------
#  Configuration
# ------------------------------------------------------------

K_SET = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

RHO_MIN   = 0.0
RHO_MAX   = 1.0
RHO_STEPS = 201

RHO_FINE_STEPS   = 1001
DELTA_FINE_STEPS = 360

N_STEPS   = 60
ALPHA     = 0.0
BETA      = 0.0
POS0      = 0
SPIN0     = 'up'

TOL_PHASE = 1e-9
TOL_POS   = 1e-7
TOL_COIN  = 1e-7
TOL_AMP   = 1e-12

TOL_EXACT = 1e-9

# Tolerance for the trivial-point test (matches classify_hit).
TOL_TRIVIAL = 1e-6

# ------------------------------------------------------------
#  Dukes exact revival table
# ------------------------------------------------------------

DUKES_EXACT = [
    (2, 0.5, 0.0, 2, "Table 2"),

    (3, 2/3,              0.0,        8,  "Table 3"),
    (3, (5 - sqrt(5))/6,  0.0,        10, "Table 3"),
    (3, 1/3,              0.0,        12, "Table 3"),
    (3, 1/3,              2*pi/3,     12, "Table 3"),
    (3, 1/3,              4*pi/3,     12, "Table 3"),

    (4, 3/4,              0.0,        6,  "Table 4"),
    (4, 1/2,              0.0,        8,  "Table 4"),
    (4, 1/2,              pi,         8,  "Table 4"),
    (4, (5 - sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, (5 + sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, 1/4,              0.0,        12, "Table 4"),
    (4, 1/4,              pi,         12, "Table 4"),
    (4, 3/4,              pi,         12, "Table 4"),
    (4, (2 - sqrt(3))/2,  pi/2,       12, "Table 4"),
    (4, (2 - sqrt(3))/2,  3*pi/2,     12, "Table 4"),

    (5,  (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),

    (8, 1/2, 0.0,    24, "Table 5"),
    (8, 1/2, pi/2,   24, "Table 5"),
    (8, 1/2, pi,     24, "Table 5"),
    (8, 1/2, 3*pi/2, 24, "Table 5"),
]

DELTA_SWEEP = [
    (0.0,      "0"),
    (2*pi/3,   "2pi/3"),
    (pi/2,     "pi/2"),
    (pi,       "pi"),
    (4*pi/3,   "4pi/3"),
    (3*pi/2,   "3pi/2"),
    (2*pi/5,   "2pi/5"),
    (4*pi/5,   "4pi/5"),
    (6*pi/5,   "6pi/5"),
    (8*pi/5,   "8pi/5"),
]

# ------------------------------------------------------------
#  Core operators
# ------------------------------------------------------------

def coin_operator(rho, alpha=0.0, beta=0.0):
    return np.array([
        [sqrt(rho),                    sqrt(1-rho)*np.exp(1j*alpha)],
        [sqrt(1-rho)*np.exp(1j*beta), -sqrt(rho)*np.exp(1j*(alpha+beta))]
    ], dtype=complex)

def shift_operator(k):
    dim = 2*k
    S = np.zeros((dim, dim), dtype=complex)
    for i in range(k):
        S[((i-1) % k)*2,     i*2]     = 1.0
        S[((i+1) % k)*2 + 1, i*2 + 1] = 1.0
    return S

def walk_operator(k, rho, alpha=0.0, beta=0.0):
    return shift_operator(k) @ np.kron(np.eye(k, dtype=complex),
                                       coin_operator(rho, alpha, beta))

def initial_state(k, pos=0, spin='up'):
    psi = np.zeros(2*k, dtype=complex)
    psi[pos*2 + (0 if spin == 'up' else 1)] = 1.0
    return psi

# ------------------------------------------------------------
#  Eigenvalue residuals (exact and projective) — reference
#  helpers, using the same iterative form as the fast path
# ------------------------------------------------------------

def exact_residual_eig(U, N):
    eigs = np.linalg.eigvals(U)
    z = np.ones_like(eigs)
    for _ in range(N):
        z *= eigs
    return float(np.max(np.abs(z - 1.0)))

def projective_residual_eig(U, N):
    eigs = np.linalg.eigvals(U)
    z = np.ones_like(eigs)
    for _ in range(N):
        z *= eigs
    z_mean = z.mean()
    if abs(z_mean) < 1e-15:
        return float(np.max(np.abs(z)))
    return float(np.max(np.abs(z - z_mean)))

# ------------------------------------------------------------
#  State-level comparison
# ------------------------------------------------------------

def same_up_to_phase(a, b, tol=TOL_PHASE):
    if np.linalg.norm(a) < tol or np.linalg.norm(b) < tol:
        return False
    idx = int(np.argmax(np.abs(b)))
    phase = a[idx] / b[idx]
    if abs(abs(phase) - 1.0) > tol:
        return False
    return np.allclose(a, phase * b, atol=tol)

def position_distribution(psi, k):
    p = np.abs(psi)**2
    return p[0::2] + p[1::2]

def coin_state(psi, k):
    psi_mat = psi.reshape(k, 2)
    return psi_mat.conj().T @ psi_mat

def same_position_distribution(pa, pb, tol=TOL_POS):
    return np.allclose(pa, pb, atol=tol)

def same_coin_state(ra, rb, tol=TOL_COIN):
    return np.allclose(ra, rb, atol=tol)

def is_localized(psi, k, tol=TOL_AMP):
    p = position_distribution(psi, k)
    return int(np.sum(p > tol)) == 1

# ------------------------------------------------------------
#  State-level scan (grid-search mode)
# ------------------------------------------------------------

def scan_one_rho(k, rho, N, alpha, beta, pos0, spin0):
    U    = walk_operator(k, rho, alpha, beta)
    psi0 = initial_state(k, pos0, spin0)
    psi  = psi0.copy()

    out = dict(full=[], pos_only=[], coin_only=[],
               both_phase=[], coin_triv=[])

    for step in range(1, N+1):
        psi = U @ psi

        full = same_up_to_phase(psi, psi0)
        pos  = same_position_distribution(
            position_distribution(psi, k),
            position_distribution(psi0, k))
        coin = same_coin_state(coin_state(psi, k),
                               coin_state(psi0, k))
        loc  = is_localized(psi, k)

        if full:
            out['full'].append(step)
        elif pos and coin:
            out['both_phase'].append(step)
        elif pos:
            out['pos_only'].append(step)
        elif coin:
            if loc:
                out['coin_triv'].append(step)
            else:
                out['coin_only'].append(step)

    return out

# ------------------------------------------------------------
#  Trivial-point test (exactly the set classify_hit labels
#  'trivial'; used to skip those points before the eigensolve).
#  delta is periodic with 2*pi in the coin, so delta == 0 and
#  delta == 2*pi are the same point; the grid uses
#  endpoint=False so 2*pi never appears.
# ------------------------------------------------------------

def is_trivial_point(rho, delta, tol=TOL_TRIVIAL):
    if abs(rho) < tol or abs(rho - 1.0) < tol:
        return True
    if abs(delta) < tol or abs(delta - pi) < tol:
        return True
    return False

# ------------------------------------------------------------
#  EXHAUSTIVE scan (v6.1, operator-level)
# ------------------------------------------------------------

def scan_exhaustive_one_point(k, rho, delta, N_max, tol=TOL_EXACT):
    """Operator-level exact/projective scan at a single (rho, delta).

    Trivial endpoint points (rho in {0,1}, delta in {0, pi}) are
    classified analytically and returned with n_exact = None and
    n_proj = None, without doing an eigensolve.  The caller
    counts them separately.

    The N-loop uses the iterative form z *= eigs (fast).  When a
    candidate is detected at step N, eigs ** N is recomputed
    exactly to confirm the detection and to pin phi_proj to
    machine precision.  This eliminates any accumulated phase
    drift from the iterative form before phi feeds the phase-
    order derivation N_exact = N_proj * ord(phi).
    """
    if is_trivial_point(rho, delta):
        return dict(n_exact=None, n_proj=None, phi_proj=None,
                    worst_exact=0.0, worst_proj=0.0, trivial=True)

    U = walk_operator(k, rho, 0.0, delta)
    eigs = np.linalg.eigvals(U)

    n_exact = None
    n_proj = None
    phi_proj = None
    worst_exact = 0.0
    worst_proj = 0.0

    # Iterative power: z_N = eigs**N.  Equivalent to eigs**N to
    # ~1e-14 at N=60 with |lambda|=1; confirmations below use
    # eigs**N exactly.
    z = np.ones_like(eigs)
    for N in range(1, N_max + 1):
        z *= eigs

        e_res = float(np.max(np.abs(z - 1.0)))
        if e_res > worst_exact:
            worst_exact = e_res
        if n_exact is None and e_res < tol:
            # Recompute exactly at the detected N (one array op,
            # only on actual detections) to remove any drift.
            z_confirm = eigs ** N
            if float(np.max(np.abs(z_confirm - 1.0))) < tol:
                n_exact = N

        z_mean = z.mean()
        if abs(z_mean) > 1e-15:
            p_res = float(np.max(np.abs(z - z_mean)))
            if p_res > worst_proj:
                worst_proj = p_res
            if n_proj is None and p_res < tol:
                # Recompute exactly to pin phi_proj.
                z_confirm = eigs ** N
                zc_mean = z_confirm.mean()
                if abs(zc_mean) > 1e-15:
                    p_res_c = float(np.max(np.abs(z_confirm - zc_mean)))
                    if p_res_c < tol:
                        n_proj = N
                        phi_proj = zc_mean / abs(zc_mean)

        if n_exact is not None and n_proj is not None:
            break

    return dict(n_exact=n_exact, n_proj=n_proj, phi_proj=phi_proj,
                worst_exact=worst_exact, worst_proj=worst_proj,
                trivial=False)

# ------------------------------------------------------------
#  Build rho / delta grids
# ------------------------------------------------------------

def build_rho_grid(k, rho_min, rho_max, rho_steps,
                   delta, include_exact=True, exact_tol=1e-6):
    grid = list(np.linspace(rho_min, rho_max, rho_steps))
    if include_exact:
        related = {k}
        if k % 2 == 0 and (k // 2) % 2 == 1:
            related.add(k // 2)
        if k % 2 == 1:
            related.add(2 * k)
        for (kk, rr, dd, _, _) in DUKES_EXACT:
            if kk not in related:
                continue
            if abs(dd - delta) > exact_tol:
                continue
            if rho_min - exact_tol <= rr <= rho_max + exact_tol:
                grid.append(rr)
    grid = sorted(set(round(x, 12) for x in grid))
    return grid

def build_delta_grid(delta_steps):
    grid = list(np.linspace(0.0, 2*pi, delta_steps, endpoint=False))
    for (d, _) in DELTA_SWEEP:
        grid.append(d)
    grid = sorted(set(round(x, 12) for x in grid))
    return grid

# ------------------------------------------------------------
#  Dukes lookup
# ------------------------------------------------------------

def is_dukes_exact(k, rho, delta, N1,
                   rho_tol=1e-4, delta_tol=1e-4, N_tol=1):
    for (kk, rr, dd, NN, src) in DUKES_EXACT:
        if kk != k:           continue
        if abs(rr - rho) > rho_tol:   continue
        if abs(dd - delta) > delta_tol: continue
        if N1 is not None and abs(NN - N1) > N_tol: continue
        return src
    return None

def snap_delta(delta, tol=1e-6):
    exact_deltas = sorted({dd for (_, _, dd, _, _) in DUKES_EXACT})
    for ed in exact_deltas:
        if abs(delta - ed) < tol:
            return ed
    return delta

# ------------------------------------------------------------
#  Snap helpers + refinement
# ------------------------------------------------------------

def snap_rho_fine(rho, max_denom=30, tol=1e-3):
    frac = Fraction(rho).limit_denominator(max_denom)
    if abs(float(frac) - rho) < tol:
        return float(frac)
    return rho

def snap_delta_fine(delta, max_denom=60, tol=1e-3):
    frac = Fraction(delta / (2*pi)).limit_denominator(max_denom)
    snapped = float(frac) * 2 * pi
    if abs(snapped - delta) < tol:
        return snapped
    return delta

def refine_candidate(k, rho, delta, N_exact, tol_eig=1e-12, tol_mat=1e-10):
    rho_s   = snap_rho_fine(rho)
    delta_s = snap_delta_fine(delta)

    U = walk_operator(k, rho_s, 0.0, delta_s)

    eigs = np.linalg.eigvals(U)
    z = eigs ** N_exact
    eig_resid = float(np.max(np.abs(z - 1.0)))

    UN = np.linalg.matrix_power(U, N_exact)
    mat_resid = float(np.linalg.norm(UN - np.eye(UN.shape[0])))

    confirmed = (eig_resid < tol_eig) and (mat_resid < tol_mat)

    return dict(confirmed=confirmed,
                rho_snap=rho_s, delta_snap=delta_s,
                eig_resid=eig_resid, mat_resid=mat_resid)

# ------------------------------------------------------------
#  Grid-search driver
# ------------------------------------------------------------

def sweep(k_list, rho_min, rho_max, rho_steps,
          N, alpha, beta, pos0, spin0, verbose=1):
    delta   = alpha + beta
    records = []

    for k in k_list:
        rho_grid = build_rho_grid(k, rho_min, rho_max, rho_steps, delta)

        if verbose >= 1:
            print(f"\n{'='*70}")
            print(f"  k = {k}   delta = {delta:.6f}   "
                  f"({len(rho_grid)} rho points, N_max = {N})")
            print(f"{'='*70}")

        for rho in rho_grid:
            r = scan_one_rho(k, rho, N, alpha, beta, pos0, spin0)
            rec = dict(k=k, rho=float(rho), delta=float(delta),
                       full=r['full'],
                       pos_only=r['pos_only'],
                       coin_only=r['coin_only'],
                       both_phase=r['both_phase'],
                       coin_triv=r['coin_triv'])
            records.append(rec)

            if verbose >= 2:
                def s(v): return "-" if not v else ",".join(map(str, v))
                print(f"  {rho:8.4f}  full={s(r['full']):<30} "
                      f"pos={s(r['pos_only']):<10} "
                      f"coin={s(r['coin_only']):<10} "
                      f"both={s(r['both_phase']):<10}")

        if verbose >= 1:
            fulls = [r for r in records if r['k'] == k and r['full']]
            trivp = [r for r in records if r['k'] == k and r['coin_triv']]
            print(f"  -> {len(fulls)} rho values give a FULL revival")
            for rec in fulls[:12]:
                print(f"       rho={rec['rho']:.6f}  N={rec['full']}")
            if len(fulls) > 12:
                print(f"       ... (+{len(fulls)-12} more)")
            if trivp:
                print(f"  -> {len(trivp)} rho values give trivial coin "
                      f"matches (localized)")

    return records

# ------------------------------------------------------------
#  Per-k worker for exhaustive_sweep (module-level so it is
#  picklable under multiprocessing's spawn start method)
# ------------------------------------------------------------

def _one_k_exhaustive(k, rho_steps, delta_steps, N_max,
                      rho_min, rho_max, verbose):
    """Run the full (rho, delta) grid for a single k."""
    delta_grid = build_delta_grid(delta_steps)

    rho_grid = build_rho_grid(k, rho_min, rho_max, rho_steps,
                              delta=0.0, include_exact=True)
    dukes_rhos = sorted({rr for (kk, rr, _, _, _) in DUKES_EXACT
                         if kk == k})
    for rr in dukes_rhos:
        if rho_min - 1e-9 <= rr <= rho_max + 1e-9:
            rho_grid = sorted(set(rho_grid + [rr]))

    n_points = len(rho_grid) * len(delta_grid)
    t0 = time.time()
    exact_hits = []
    proj_hits = []
    trivial_count = 0
    worst_exact = 0.0
    worst_exact_at = None

    for rho in rho_grid:
        for delta in delta_grid:
            r = scan_exhaustive_one_point(k, rho, delta, N_max)

            if r['trivial']:
                trivial_count += 1
                continue

            if r['n_exact'] is not None:
                exact_hits.append((rho, delta, r['n_exact']))
            if r['n_proj'] is not None:
                proj_hits.append((rho, delta, r['n_proj'],
                                  r['phi_proj']))

            if r['worst_exact'] > worst_exact:
                worst_exact = r['worst_exact']
                worst_exact_at = (rho, delta)

    # ... grid sweep ...
    elapsed_sweep = time.time() - t0

    # Refinement (interior points only -- trivial points were
    # skipped above and never enter exact_hits).
    t_refine = time.time()
    refined = []
    unconfirmed = []
    for (rho, delta, N) in exact_hits:
        r = refine_candidate(k, rho, delta, N)
        if r['confirmed']:
            refined.append(dict(rho=rho, delta=delta, N=N,
                                rho_snap=r['rho_snap'],
                                delta_snap=r['delta_snap'],
                                eig_resid=r['eig_resid'],
                                mat_resid=r['mat_resid']))
        else:
            unconfirmed.append(dict(rho=rho, delta=delta, N=N,
                                    rho_snap=r['rho_snap'],
                                    delta_snap=r['delta_snap'],
                                    eig_resid=r['eig_resid'],
                                    mat_resid=r['mat_resid']))

    elapsed_refine = time.time() - t_refine
    elapsed = elapsed_sweep + elapsed_refine

    if verbose:
        print(f"  k={k:>2}:  "
              f"candidates={len(exact_hits):>4}  "
              f"refined={len(refined):>4}  "
              f"unconfirmed={len(unconfirmed):>3}  "
              f"proj={len(proj_hits):>4}  "
              f"trivial_skipped={trivial_count:>5}  "
              f"worst_exact={worst_exact:.2e}  "
              f"(sweep {elapsed_sweep:.1f}s + refine {elapsed_refine:.1f}s "
              f"= {elapsed:.1f}s)", flush=True)

    return dict(
        rho_grid_size=len(rho_grid),
        delta_grid_size=len(delta_grid),
        n_points=n_points,
        trivial_skipped=trivial_count,
        elapsed=elapsed,
        exact_hits=exact_hits,
        refined_hits=refined,
        unconfirmed=unconfirmed,
        proj_hits=proj_hits,
        worst_exact=worst_exact,
        worst_exact_at=worst_exact_at,
    )

# ------------------------------------------------------------
#  EXHAUSTIVE driver (parallelised across k)
# ------------------------------------------------------------

def _available_cpus():
    """Available cores, respecting cgroup limits where the OS
    exposes them.  Falls back to os.cpu_count()."""
    try:
        return len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        return os.cpu_count() or 1

def exhaustive_sweep(k_list, rho_steps, delta_steps, N_max,
                     rho_min=RHO_MIN, rho_max=RHO_MAX,
                     verbose=True, n_workers=None):
    delta_grid = build_delta_grid(delta_steps)
    rho_grid_base = np.linspace(rho_min, rho_max, rho_steps)

    if n_workers is None:
        n_workers = min(len(k_list), _available_cpus())

    print("="*78)
    print("EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion (v6.1)")
    print("="*78)
    print(f"  k values          : {k_list}")
    print(f"  rho grid          : {rho_steps} points in [{rho_min}, {rho_max}]")
    print(f"  delta grid        : {len(delta_grid)} points in [0, 2pi)  "
          f"(requested {delta_steps}, plus Dukes exact values)")
    print(f"  N max             : {N_max}")
    print(f"  exact tolerance   : {TOL_EXACT:.1e}")
    print(f"  trivial tolerance : {TOL_TRIVIAL:.1e}  "
          f"(rho in {{0,1}}, delta in {{0,pi}} skipped)")
    print(f"  BLAS threads      : OMP/OPENBLAS/MKL = 1 per worker")
    print(f"  cpu_count         : {os.cpu_count()}")
    print(f"  sched_getaffinity : {_available_cpus()}")
    print(f"  workers           : {n_workers}")
    print()

    results = {}
    t_all = time.time()

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {
            ex.submit(_one_k_exhaustive,
                      k, rho_steps, delta_steps, N_max,
                      rho_min, rho_max, verbose): k
            for k in k_list
        }
        for fut in as_completed(futures):
            k = futures[fut]
            try:
                results[k] = fut.result()
            except Exception as e:
                print(f"  k={k}: WORKER FAILED: {e}", flush=True)
                raise

    if verbose:
        print(f"\n  total wall time: {time.time() - t_all:.1f}s")

    return results, rho_grid_base, delta_grid

def classify_hit(rho, delta, k, N, tol=TOL_TRIVIAL):
    if abs(delta) < tol or abs(delta - pi) < tol:
        return 'trivial'
    if abs(rho) < tol or abs(rho - 1.0) < tol:
        return 'trivial'
    src = is_dukes_exact(k, rho, delta, N)
    if src is not None:
        return 'dukes'
    return 'new'

def print_exhaustive_report(results, k_list, out_txt=None):
    lines = []
    def p(s=""):
        print(s)
        lines.append(s)

    p()
    p("="*78)
    p("EXHAUSTIVE SWEEP — SUMMARY (v6.1)")
    p("="*78)

    for k in k_list:
        r = results[k]
        p()
        p(f"k = {k}")
        p(f"  grid: {r['rho_grid_size']} rho x {r['delta_grid_size']} delta "
          f"= {r['n_points']:,} points  "
          f"({r['trivial_skipped']:,} trivial endpoints skipped)")

        # --- two-way split of refined hits (list of dicts).
        # Trivial points are excluded upstream by construction, so
        # 'refined_hits' contains only Dukes or new entries.
        dukes_hits = []
        new_hits   = []
        for hit in r['refined_hits']:
            rho   = hit['rho']
            delta = hit['delta']
            N     = hit['N']
            kind  = classify_hit(rho, delta, k, N)
            if kind == 'dukes':
                dukes_hits.append((rho, delta, N))
            elif kind == 'new':
                new_hits.append((rho, delta, N))
            # 'trivial' cannot occur here: trivial points are
            # skipped before eigensolve in _one_k_exhaustive.

        p(f"  confirmed revivals found: {len(r['refined_hits'])}  "
          f"({len(dukes_hits)} Dukes, "
          f"{len(new_hits)} new; "
          f"{r['trivial_skipped']:,} trivial endpoints skipped)")

        # --- warn about unconfirmed candidates (list of dicts) ---
        if r['unconfirmed']:
            p(f"  WARNING: {len(r['unconfirmed'])} unconfirmed candidates "
              f"(passed grid tolerance, failed refinement)")
            for c in r['unconfirmed'][:5]:
                p(f"    rho={c['rho']:.6f}  delta={c['delta']:.6f}  "
                  f"N={c['N']}  "
                  f"eig={c['eig_resid']:.2e}  mat={c['mat_resid']:.2e}")
            if len(r['unconfirmed']) > 5:
                p(f"    ... (+{len(r['unconfirmed'])-5} more)")

        # --- detail listing ---
        if new_hits:
            by_n = {}
            for (rho, delta, N) in new_hits:
                by_n.setdefault(N, []).append((rho, delta))
            p(f"  NEW exact revivals:")
            for N in sorted(by_n):
                pts = by_n[N]
                p(f"    N = {N:>3}:  {len(pts)} point(s)")
                for (rho, delta) in pts[:8]:
                    p(f"      rho={rho:.6f}  delta={delta:.6f}")
                if len(pts) > 8:
                    p(f"      ... (+{len(pts)-8} more)")

        elif dukes_hits:
            by_n = {}
            for (rho, delta, N) in dukes_hits:
                by_n.setdefault(N, []).append((rho, delta))
            p(f"  Dukes-tabulated exact revivals:")
            for N in sorted(by_n):
                pts = by_n[N]
                p(f"    N = {N:>3}:  {len(pts)} point(s)")
                for (rho, delta) in pts[:6]:
                    p(f"      rho={rho:.6f}  delta={delta:.6f}")
                if len(pts) > 6:
                    p(f"      ... (+{len(pts)-6} more)")

        else:
            p(f"  (no Dukes-tabulated or new exact revivals on this grid)")

        # --- sensitivity witness ---
        p(f"  worst exact residual over grid: "
          f"{r['worst_exact']:.2e}")
        if r['worst_exact_at']:
            rho_w, delta_w = r['worst_exact_at']
            p(f"    at rho={rho_w:.6f}, delta={delta_w:.6f}")

        p(f"  projective revivals found: {len(r['proj_hits'])}")

    if out_txt:
        with open(out_txt, 'w') as f:
            f.write("\n".join(lines))
        print(f"\nReport written to {out_txt}")


# ------------------------------------------------------------
#  v4 compact summary (unchanged)
# ------------------------------------------------------------

def _format_harmonics(harms):
    if not harms:
        return "-"
    if len(harms) <= 3:
        return ", ".join(str(x) for x in harms)
    d = harms[1] - harms[0]
    is_ap = all(harms[i+1] - harms[i] == d for i in range(len(harms)-1))
    if is_ap:
        return f"{harms[0]}, {harms[0]+d}, ..., {harms[-1]}"
    return f"{harms[0]}, {harms[1]}, ..., {harms[-1]}"

def print_compact_summary(records, k_list, skip_endpoints=True,
                          flag_new=False):
    print("\n" + "#"*70)
    print("#  COMPACT SUMMARY — FULL revivals (interior rho only)")
    if flag_new:
        print("#  (*)  = not in Dukes' tables")
    print("#"*70)

    for k in k_list:
        if k == 2:
            print(f"\nk = 2")
            print("  (continuous family: all rho in [0,1] give "
                  "N1 = 2 at delta = 0)")
            continue

        recs = [r for r in records if r['k'] == k and r['full']]
        if skip_endpoints:
            recs = [r for r in recs
                    if 1e-6 < r['rho'] < 1 - 1e-6]

        print(f"\nk = {k}")
        if not recs:
            print("  (no interior full revivals)")
            continue

        print(f"  {'':1} {'rho':>12}  {'N1':>4}  {'harmonics':>24}")
        for r in sorted(recs, key=lambda x: x['rho']):
            n1 = r['full'][0]
            harms = [n for n in r['full'] if n != n1]
            marker = " "
            if flag_new:
                src = is_dukes_exact(k, r['rho'], r['delta'], n1)
                marker = " " if src else "*"
            print(f"  {marker} {r['rho']:12.6f}  {n1:>4}  "
                  f"{_format_harmonics(harms):>24}")

# ------------------------------------------------------------
#  CSV export (v4, unchanged)
# ------------------------------------------------------------

def write_csv(records, filename):
    with open(filename, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['k', 'rho', 'delta', 'full', 'pos_only',
                    'coin_only', 'both_phase', 'coin_triv'])
        for r in records:
            def j(v): return ";".join(map(str, v)) if v else ""
            w.writerow([r['k'], f"{r['rho']:.10f}", f"{r['delta']:.10f}",
                        j(r['full']), j(r['pos_only']),
                        j(r['coin_only']), j(r['both_phase']),
                        j(r['coin_triv'])])

# ------------------------------------------------------------
#  Main
# ------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(
        description="Sweep rho/delta for Dukes quantum walk revivals "
                    "on k-cycles.")
    parser.add_argument('--k', type=str, default=None,
                        help='comma-separated k values (default: K_SET)')
    parser.add_argument('--k-all', action='store_true',
                        help='explicit K_SET (2..16)')
    parser.add_argument('--rho-min',   type=float, default=RHO_MIN)
    parser.add_argument('--rho-max',   type=float, default=RHO_MAX)
    parser.add_argument('--rho-steps', type=int,   default=RHO_STEPS)
    parser.add_argument('--N',         type=int,   default=N_STEPS)
    parser.add_argument('--alpha',     type=float, default=ALPHA)
    parser.add_argument('--beta',      type=float, default=BETA)
    parser.add_argument('--pos0',      type=int,   default=POS0)
    parser.add_argument('--spin0', choices=['up', 'down'], default=SPIN0)
    parser.add_argument('--verbose', type=int, default=1)
    parser.add_argument('--csv', type=str, default=None)
    parser.add_argument('--delta-sweep', action='store_true')
    parser.add_argument('--flag-new', action='store_true')

    # v5/v6 flags
    parser.add_argument('--exhaustive', action='store_true',
                        help='dense (rho, delta) grid, operator-level '
                             'exact criterion, quantify negative results')
    parser.add_argument('--rho-fine', type=int, default=RHO_FINE_STEPS,
                        help=f'rho grid points for --exhaustive '
                             f'(default {RHO_FINE_STEPS})')
    parser.add_argument('--delta-fine', type=int, default=DELTA_FINE_STEPS,
                        help=f'delta grid points for --exhaustive '
                             f'(default {DELTA_FINE_STEPS})')
    parser.add_argument('--report', type=str, default=None,
                        help='text file for exhaustive-mode report')
    parser.add_argument('--workers', type=int, default=None,
                        help='worker processes for --exhaustive '
                             '(default: min(len(k), available_cpus()))')

    args, _ = parser.parse_known_args()

    if args.k_all:
        k_list = K_SET
    elif args.k:
        k_list = [int(x) for x in args.k.split(',')]
    else:
        k_list = K_SET

    # ---- exhaustive mode ----
    if args.exhaustive:
        results, _, _ = exhaustive_sweep(
            k_list,
            rho_steps=args.rho_fine,
            delta_steps=args.delta_fine,
            N_max=args.N,
            n_workers=args.workers,
        )
        print_exhaustive_report(results, k_list, out_txt=args.report)
        return

    # ---- v4 delta-sweep mode ----
    if args.delta_sweep:
        for (delta, label) in DELTA_SWEEP:
            print("\n" + "="*70)
            print(f"  DELTA SWEEP : delta = {label}  "
                  f"(numeric = {delta:.10f})")
            print("="*70)
            alpha, beta = 0.0, delta
            beta = snap_delta(beta)

            records = sweep(k_list, args.rho_min, args.rho_max,
                            args.rho_steps, args.N, alpha, beta,
                            args.pos0, args.spin0, verbose=0)
            print_compact_summary(records, k_list,
                                  skip_endpoints=True,
                                  flag_new=args.flag_new)
            if args.csv:
                out = args.csv.replace('.csv',
                                       f"_{label.replace('/','_')}.csv")
                write_csv(records, out)
                print(f"CSV written to {out}")
        return

    # ---- v4 single-delta mode ----
    raw_delta    = args.alpha + args.beta
    snapped      = snap_delta(raw_delta)
    if abs(snapped - raw_delta) > 1e-12:
        args.beta = snapped - args.alpha

    print("="*70)
    print("EXP032b.py v6.1  —  Dukes quantum walk revival sweep")
    print("="*70)
    print(f"k values    : {k_list}")
    print(f"rho range   : {args.rho_min} .. {args.rho_max} "
          f"({args.rho_steps} points + Dukes exact values)")
    print(f"steps N     : {args.N}")
    if abs(snapped - raw_delta) > 1e-12:
        print(f"NOTE: delta = {raw_delta} snapped to Dukes exact "
              f"value {snapped}")
    print(f"alpha, beta : {args.alpha}, {args.beta}  "
          f"(delta = {args.alpha + args.beta})")
    print(f"initial     : |{args.pos0}, {args.spin0}>")
    print("="*70)

    records = sweep(k_list, args.rho_min, args.rho_max, args.rho_steps,
                    args.N, args.alpha, args.beta,
                    args.pos0, args.spin0, verbose=args.verbose)

    print_compact_summary(records, k_list,
                          skip_endpoints=True,
                          flag_new=args.flag_new)

    if args.csv:
        write_csv(records, args.csv)
        print(f"\nCSV written to {args.csv}")

if __name__ == "__main__":
    main()

Overwriting EXP032b.py


## Test Coarse scan - rho=200 grid - on non-revival k values
Expected : no revivals found other than trivial rho = (0,1).

Full scan on coarse grid with 2 workers on k in {7,9,11,12,13,14,15,16}

Timing : supplied inline




In [2]:
!python EXP032b.py --exhaustive --k 7,9,11,12,13,14,15,16 \
    --rho-fine 201 --delta-fine 72 \
    --report exhaustive_quick.txt --workers 2

EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion (v6.1)
  k values          : [7, 9, 11, 12, 13, 14, 15, 16]
  rho grid          : 201 points in [0.0, 1.0]
  delta grid        : 76 points in [0, 2pi)  (requested 72, plus Dukes exact values)
  N max             : 60
  exact tolerance   : 1.0e-09
  trivial tolerance : 1.0e-06  (rho in {0,1}, delta in {0,pi} skipped)
  BLAS threads      : OMP/OPENBLAS/MKL = 1 per worker
  cpu_count         : 2
  sched_getaffinity : 2
  workers           : 2

  k= 7:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped=  550  worst_exact=2.00e+00  (sweep 87.4s + refine 0.0s = 87.4s)
  k= 9:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped=  550  worst_exact=2.00e+00  (sweep 90.5s + refine 0.0s = 90.5s)
  k=11:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped=  550  worst_exact=2.00e+00  (sweep 90.2s + refine 0.0s = 90.2s)
  k=12:  candidates=   0  refined=   0  unconfirmed

## Fine scan on full 1000 x 360 grid for all non-reviving k values

Full scan on 1000 x 360 grid with 2 workers on k in {7,9,11,12,13,14,15,16} for paper claim that there are no non-trivial revivals for these values

Any revivals found will be trivial (rho = 0 or rho = 1)

This takes significantly more time to run ~ 1600s per k value on 2 core Mac Air


In [3]:
!python EXP032b.py --exhaustive --k 7,9,11,12,13,14,15,16 \
    --report exhaustive_report_k_no_revivals.txt --workers 2

EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion (v6.1)
  k values          : [7, 9, 11, 12, 13, 14, 15, 16]
  rho grid          : 1001 points in [0.0, 1.0]
  delta grid        : 360 points in [0, 2pi)  (requested 360, plus Dukes exact values)
  N max             : 60
  exact tolerance   : 1.0e-09
  trivial tolerance : 1.0e-06  (rho in {0,1}, delta in {0,pi} skipped)
  BLAS threads      : OMP/OPENBLAS/MKL = 1 per worker
  cpu_count         : 2
  sched_getaffinity : 2
  workers           : 2

  k= 7:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped= 2718  worst_exact=2.00e+00  (sweep 1677.9s + refine 0.0s = 1677.9s)
  k= 9:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped= 2718  worst_exact=2.00e+00  (sweep 1756.0s + refine 0.0s = 1756.0s)
  k=11:  candidates=   0  refined=   0  unconfirmed=  0  proj=   0  trivial_skipped= 2718  worst_exact=2.00e+00  (sweep 1413.2s + refine 0.0s = 1413.2s)
  k=12:  candidates=   0  refined=  

## Exhaustive search over $(\rho, \delta)$ for $k \in 2,3,4,5,6,8,10$

This cell runs `EXP032b.py --exhaustive`, which switches to:

- a dense $(\rho, \delta)$ grid (default $1001 \times 360$, i.e.
  about $3.6 \times 10^5$ points per $k$),
- the operator-level exact criterion $U^N = I$, computed via the
  eigenvalue residual $\max_j |\lambda_j^N - 1|$,
- $N$ from $1$ to $60$ at each grid point.

For each $k$ the output reports:

- every exact revival found, as $(k, \rho, \delta, N_{\text{exact}})$;
- the **worst exact residual over the whole grid**, for $k$ with no
  revival. This is the sensitivity witness: a value of order
  $10^{-1}$ means the search was genuinely sensitive and the
  negative result is meaningful; a value near the $10^{-9}$
  tolerance would mean the grid was flirting with false positives
  and a finer grid might find something.

Runtime is expected to be < 1600s per k value

### What to check

- For $k \in \{2, 3, 4, 5, 6, 8, 10\}$: the exact revivals should
  reproduce the `ONE_D_FULL` catalogue from `EXP033.ipynb`.
- For $k \in \{7, 9, 11, 12, 13, 14, 15, 16\}$: no exact revivals,
  with worst residual of order $10^{-1}$.  Done in previous cell
- The k=2, $\rho=1/2$, $\delta=\pi/2$ correction: `rho=0.500000`
  should appear at `N=24` in the $\delta \approx \pi/2$ block. If
  it appears at `N=6` only, the criterion is projective, not exact,
  and the mode is misconfigured.

In [ ]:
!python EXP032b.py --exhaustive --k 2,3,4,5,6,8,10 \
    --report exhaustive_report_k_known_revivals.txt --workers 2

EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion (v6.1)
  k values          : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  rho grid          : 1001 points in [0.0, 1.0]
  delta grid        : 360 points in [0, 2pi)  (requested 360, plus Dukes exact values)
  N max             : 60
  exact tolerance   : 1.0e-09
  trivial tolerance : 1.0e-06  (rho in {0,1}, delta in {0,pi} skipped)
  BLAS threads      : OMP/OPENBLAS/MKL = 1 per worker
  cpu_count         : 2
  sched_getaffinity : 2
  workers           : 2

  k= 2:  candidates=   2  refined=   2  unconfirmed=  0  proj=   2  trivial_skipped= 2718  worst_exact=2.00e+00  (sweep 1119.8s + refine 0.0s = 1119.8s)
  k= 3:  candidates=  12  refined=   8  unconfirmed=  4  proj=  12  trivial_skipped= 2730  worst_exact=2.00e+00  (sweep 1151.5s + refine 0.0s = 1151.5s)
  k= 4:  candidates=   4  refined=   2  unconfirmed=  2  proj=   4  trivial_skipped= 2728  worst_exact=2.00e+00  (sweep 1212.1s + refine 0.0s = 1212.1s)
  k= 5:  candi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.download('exhaustive_report.txt'

In [ ]:
from typing import Text
import zipfile
from pathlib import Path

outputs = ["exhaustive_report_k_no_revivals.txt",
           "exhaustive_report_k_known_revivals.txt",
           "exhaustive_quick.txt"]

with zipfile.ZipFile("EXP032b_v6_outputs.zip", "w",
                     zipfile.ZIP_DEFLATED) as z:
    for f in outputs:
        if Path(f).exists():
            z.write(f)
            print(f"added {f}")
        else:
            print(f"WARNING: {f} not found")

print()
print(f"EXP033b_v2_outputs.zip written "
      f"({Path('EXP032b_v2_outputs.zip').stat().st_size} bytes)")

In [ ]:
try:
    from google.colab import files
    files.download("EXP032b_v6_outputs.zip")
except ImportError:
    print("Not running in Colab; zip is in the working directory.")

## Re-classification of the exhaustive hits

The exhaustive run above detects all exact revivals on the grid, but
the split between "trivial", "Dukes" and "new" depends on the
`classify_hit` function that was live at the time the run started.
This cell re-classifies the hits with the current `classify_hit`, so
the split is correct regardless of when the `%%writefile` cell was
last executed. It does not re-run the search.